In [25]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
candidates.extend(parent / "PatchTrack" for parent in [cwd, *cwd.parents])
repo_root = next(
    (path for path in candidates if (path / "PatchTrack.py").is_file() and (path / "analyzer").is_dir()),
    None,
)
if repo_root is None:
    raise FileNotFoundError(f"Could not locate the PatchTrack project from {cwd}")
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

print(f"PatchTrack root: {repo_root}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
PatchTrack root: C:\theory_and_practice_of_software_design\PatchTrack


In [26]:
# No system libmagic dependency is required for classification.

In [27]:
from analyzer.main import PatchTrack
from analyzer import common
from analyzer import constant

In [28]:
# Classification reads only the extracted local patches; no GitHub token is needed.
data = []

In [29]:
print("Working directory:", os.getcwd())
print("Patch directory exists:", os.path.isdir("data/patches"))

Working directory: C:\theory_and_practice_of_software_design\PatchTrack
Patch directory exists: True


Create a magic cookie. This is needed during classification

In [30]:
common.verbose_mode = False

In [31]:
data = []

In [32]:
experiment = PatchTrack([])

In [33]:
pr_project_pair_records = experiment.build_pr_project_pairs()
pr_project_pair = [
    (pr, project)
    for pair in pr_project_pair_records
    for pr, project in pair.items()
]
print(f"PR/repository pairs: {len(pr_project_pair)}")

PR/repository pairs: 285


In [34]:
# take a peek of the the data
pr_project_pair[0:4]

[('79', '23-OSSCA-python-mysql-replication/python-mysql-replication'),
 ('214', '2DegreesInvesting/tiltIndicatorAfter'),
 ('13', '40LF0/couple_project_backend'),
 ('50', '4dn-dcic/foursight-core')]

In [35]:

print(len(pr_project_pair))

285


In [36]:
print("First five PR/repository pairs:", pr_project_pair[:5])

First five PR/repository pairs: [('79', '23-OSSCA-python-mysql-replication/python-mysql-replication'), ('214', '2DegreesInvesting/tiltIndicatorAfter'), ('13', '40LF0/couple_project_backend'), ('50', '4dn-dcic/foursight-core'), ('14', 'AdamAkiva/dashboard')]


In [38]:
# Classify the local patch set and build the DataFrames used below.
experiment.classify(pr_project_pair[:5])
experiment.create_dataframes()
patch_classes = experiment.get_df_patch_classes()
classification_cache_version = 2

In [14]:
# Run once after classification to save it. In a later kernel, run setup cells,
# skip the expensive classify cell, then run this cell to restore the results.
import pickle

CLASSIFICATION_CACHE_VERSION = 2
classification_cache = repo_root / "output" / "patchtrack_classification_cache.pkl"
classification_cache.parent.mkdir(parents=True, exist_ok=True)

if experiment.df_patch_classes is not None:
    if globals().get("classification_cache_version") != CLASSIFICATION_CACHE_VERSION:
        raise RuntimeError(
            "Classification results were not recomputed with the current "
            "classifier. Run the classify cell before saving this cache."
        )
    classification_state = {
        "cache_version": CLASSIFICATION_CACHE_VERSION,
        "result_dict": experiment.result_dict,
        "pr_classifications": experiment.pr_classifications,
        "df_files_classes": experiment.df_files_classes,
        "df_patch_classes": experiment.df_patch_classes,
        "df_patches": experiment.df_patches,
    }
    with classification_cache.open("wb") as cache_file:
        pickle.dump(classification_state, cache_file)
    print(f"Saved classification cache to {classification_cache}")
else:
    if not classification_cache.is_file():
        raise FileNotFoundError(
            f"No classification cache found at {classification_cache}. "
            "Run the classify cell first to create it."
        )
    with classification_cache.open("rb") as cache_file:
        classification_state = pickle.load(cache_file)
    if classification_state.get("cache_version") != CLASSIFICATION_CACHE_VERSION:
        raise RuntimeError(
            "The classification cache is outdated and predates the corrected "
            "n-gram matching. Run the classify cell, then run this cell to "
            "create a new cache."
        )
    experiment.result_dict = classification_state["result_dict"]
    experiment.pr_classifications = classification_state["pr_classifications"]
    experiment.df_files_classes = classification_state["df_files_classes"]
    experiment.df_patch_classes = classification_state["df_patch_classes"]
    experiment.df_patches = classification_state["df_patches"]
    print(f"Loaded classification cache from {classification_cache}")

patch_classes = experiment.get_df_patch_classes()

Saved classification cache to C:\theory_and_practice_of_software_design\PatchTrack\output\patchtrack_classification_cache.pkl


In [39]:
print("Number of PR/project pairs:", len(pr_project_pair[:5]))
print("First 5 pairs:", pr_project_pair[:5])
print("Patch directory:", experiment.repo_dir_files)

Number of PR/project pairs: 5
First 5 pairs: [('79', '23-OSSCA-python-mysql-replication/python-mysql-replication'), ('214', '2DegreesInvesting/tiltIndicatorAfter'), ('13', '40LF0/couple_project_backend'), ('50', '4dn-dcic/foursight-core'), ('14', 'AdamAkiva/dashboard')]
Patch directory: data/patches/


In [40]:
patch_classes = experiment.get_df_patch_classes()
if patch_classes is None:
    raise RuntimeError("Classification did not produce patch-classification data")
Path("output").mkdir(exist_ok=True)

PA = patch_classes[patch_classes['Patch Classification'] == "PA"]
PA = PA.drop_duplicates(subset=['Pull Request', 'Repo'])
PA.to_csv(f'output/N{constant.NGRAM_SIZE}_PA_PATCH.csv', index=False)
PA.shape

(1, 7)

In [41]:
NE = patch_classes[patch_classes['Patch Classification'] == 'NE']
NE = NE.drop_duplicates(subset=['Pull Request', 'Repo'])
NE.to_csv(f'output/N{constant.NGRAM_SIZE}_NE_PATCH.csv', index=False)
NE.shape

(3, 7)

In [42]:
patch_classes = experiment.get_df_patch_classes()
PN = patch_classes[patch_classes['Patch Classification'] == 'PN']
PN = PN.drop_duplicates(subset=['Pull Request', 'Repo'])
PN.to_csv(f'output/N{constant.NGRAM_SIZE}_PN_PATCH.csv', index=False)
PN.shape

(1, 7)

In [43]:
d = experiment.get_df_file_classes()
if d is None:
    raise RuntimeError("Classification did not produce file-classification data")
d.to_csv(f'output/N{constant.NGRAM_SIZE}_ALL_PATCH_WITH_DUPLICATES.csv', index=False)
d.shape

(949, 13)

In [44]:
df = experiment.get_df_file_classes()
if df is None:
    raise RuntimeError("Classification did not produce file-classification data")
df.drop_duplicates(subset=['Pull Request', 'Repo'], inplace=True)
df.to_csv(f'output/N{constant.NGRAM_SIZE}_ALL_PATCH.csv', index=False)
df.shape

(5, 13)

In [45]:
file_classes = experiment.get_df_file_classes()
print(file_classes["File Classification"].value_counts(dropna=False))
display(
    file_classes.loc[
        file_classes["File Classification"].eq("PA"),
        ["Pull Request", "Repo", "File Path", "GitHub Patch Path", "Similarity (%)"]
    ]
)

File Classification
NOT EXISTING    3
PN              1
PA              1
Name: count, dtype: int64


,Pull Request,Repo,File Path,GitHub Patch Path,Similarity (%)
63,50,4dn-dcic/foursight-core,data/patches/4dn-dcic/foursight-core\50\chatgp...,data/patches/4dn-dcic/foursight-core\50\github...,28.57


In [46]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Bar(
    x=['PA'],
    y=[int(patch_classes['Patch Classification'].eq('PA').sum())],
    name="Patch Applied (PA)",
    marker_color='#00A0A0'
))
fig.add_trace(go.Bar(
    x=['PN'],
    y=[int(patch_classes['Patch Classification'].eq('PN').sum())],
    name="Patch Not Applied (PN)",
    marker_color='#082a54'
))
fig.add_trace(go.Bar(
    x=['NE'],
    y=[int(patch_classes['Patch Classification'].eq('NE').sum())],
    name="Non-Existent (NE)",
    marker_color='#a00000'
))
fig.add_trace(go.Bar(
    x=['CC'],
    y=[int(patch_classes['Patch Classification'].eq('CC').sum())],
    name="Cannot Classify",
    marker_color='purple'
))
fig.add_trace(go.Bar(
    x=['ERROR'],
    y=[int(patch_classes['Patch Classification'].eq('ERROR').sum())],
    name="Error",
    marker_color='#A05000'
))
fig.update_layout(
    xaxis_title="Classifications",
    yaxis_title="Frequency",
    plot_bgcolor= "rgba(0, 0, 0, 0)",
    font_color="black",
    font_size=12,
    height=350,
    width=650,
    bargap=0.45
)
fig.update_xaxes(
    mirror=True,
    ticks='outside',
    # showline=True,
    linecolor='black',
    # gridcolor='lightgrey'
    # orientation='h'
)
fig.update_yaxes(
    mirror=True,
    ticks='outside',
    # showline=True,
    linecolor='black',
    gridcolor='lightgrey',
    # orientation='h'
    # tickangle=330
)
fig.update_layout(
    # autosize=True,
    # try to play around height and width values 
    legend=dict(
    yanchor="top",
    y=0.95,
    xanchor="left",
    x=0.61,
    bordercolor="gray",
    borderwidth=1,
    itemsizing="constant", 
    itemwidth=30,
    font=dict(
            # family="Courier",
            size=12,
        ),
))
fig.show()
# fig.write_image("classification.pdf", width=650, height=350)


In [47]:
unique_prs = patch_classes.drop_duplicates(subset=["Pull Request", "Repo"])
unique_prs["Patch Classification"].value_counts()

Patch Classification
NE    3
PA    1
PN    1
Name: count, dtype: int64

In [48]:
# Create the bar chart and modify as per the instructions
fig = go.Figure(go.Bar(
            x = [6, 9, 9, 10, 11, 19],
            y = ['Internal Improvements and Documentation',
                 'Clarification and Correction',
                 'Technical Limitations and Preferences',
                 'Specific Functionality Enhancement',
                 'Methodological Guidance',
                 'Adaptation and Tailored Solutions'
                 ],
            orientation='h',
            marker_color='#082a54'))

# Remove gray background and set background to white
fig.update_xaxes(
    mirror=True,
    ticks='outside',
    # showline=True,
    linecolor='black',
    # gridcolor='lightgrey'
    # orientation='h'
)
fig.update_yaxes(
    mirror=True,
    ticks='outside',
    # showline=True,
    linecolor='black',
    gridcolor='lightgrey',
    # orientation='h'
    # tickangle=330
)

# Add axis labels
fig.update_layout(
    # xaxis_title="Number of Animals",  # Label for X-axis
    # yaxis_title="Types of Animals",  # Label for Y-axis
    xaxis_title="Frequency",
    yaxis_title="Theme",
    plot_bgcolor= "rgba(0, 0, 0, 0)",
    font_color="black",
    font_size=12,
    height=300,
    width=600,
    # bargap=0.45
)

# Show the figure
fig.show()
# fig.write_image("pn_themes.pdf", width=600, height=300)

In [49]:
# Create the bar chart and modify as per the instructions
fig = go.Figure(go.Bar(
            x = [9, 11, 25, 61],
            y = [ 'Debugging and Optimization Strategies',
                 'Education and Knowledge Sharing',
                 'Documentation, Communication, & Translation',
                 'Conceptual Guidance & Theoretical Advice',
                 ],
            orientation='h',
            marker_color='#a00000'))

# Remove gray background and set background to white
fig.update_xaxes(
    mirror=True,
    ticks='outside',
    # showline=True,
    linecolor='black',
    # gridcolor='lightgrey'
    # orientation='h'
)
fig.update_yaxes(
    mirror=True,
    ticks='outside',
    # showline=True,
    linecolor='black',
    gridcolor='lightgrey',
    # orientation='h'
    # tickangle=330
)

# Add axis labels
fig.update_layout(
    # xaxis_title="Number of Animals",  # Label for X-axis
    # yaxis_title="Types of Animals",  # Label for Y-axis
    xaxis_title="Frequency",
    yaxis_title="Theme",
    plot_bgcolor= "rgba(0, 0, 0, 0)",
    font_color="black",
    font_size=12,
    height=300,
    width=600,
    # bargap=0.45
)

# Show the figure
fig.show()
# Export with fig.write_image(...) if Kaleido is installed.